In [2]:
import cv2
import numpy as np
import pyttsx3
import threading
import time

from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from cvzone.HandTrackingModule import HandDetector
from collections import Counter


# ==========================================
# HAND SIGN DETECTOR
# ==========================================

class HandSignDetector:

    def __init__(
        self,
        model_path="sign_alphabet_mob_model.keras",
        class_indices_path="class_indices_mob.npy"
    ):

        print("Loading MobileNetV2 model...")

        self.model = keras.models.load_model(
            model_path,
            compile=False
        )

        print("Model loaded successfully!")


        class_indices = np.load(
            class_indices_path,
            allow_pickle=True
        ).item()


        self.idx_to_class = {
            v:k for k,v in class_indices.items()
        }


        print("\nClass Mapping:")
        print(self.idx_to_class)



        self.detector = HandDetector(
            maxHands=2,
            detectionCon=0.7
        )


        self.img_size = 224


        # prediction smoothing
        self.prediction_buffer = []


        # voice
        self.last_spoken = ""



    # ==========================================
    # VOICE
    # ==========================================

    def speak(self, text):

        # same letter should not repeat continuously
        if text == self.last_spoken:
            return


        self.last_spoken = text


        def say():

            try:

                engine = pyttsx3.init()

                engine.say(text)

                engine.runAndWait()

                engine.stop()


            except Exception as e:

                print("Voice error:", e)



        threading.Thread(
            target=say,
            daemon=True
        ).start()



    # ==========================================
    # PREPROCESS
    # ==========================================

    def preprocess_image(self, img_crop):


        canvas = np.zeros(
            (224,224,3),
            dtype=np.uint8
        )


        h,w = img_crop.shape[:2]


        if h == 0 or w == 0:

            return None,None



        scale = min(
            224/w,
            224/h
        )


        new_w = int(w*scale)
        new_h = int(h*scale)



        resized = cv2.resize(
            img_crop,
            (new_w,new_h)
        )


        x_offset = (224-new_w)//2
        y_offset = (224-new_h)//2



        canvas[
            y_offset:y_offset+new_h,
            x_offset:x_offset+new_w
        ] = resized



        processed = preprocess_input(
            canvas.astype(np.float32)
        )


        processed = np.expand_dims(
            processed,
            axis=0
        )


        return processed, canvas



    # ==========================================
    # PREDICT
    # ==========================================

    def predict(self,img):

        try:

            result = self.detector.findHands(
                img,
                draw=False
            )


            if isinstance(result, tuple):

                hands = result[0]
                img = result[1]

            else:

                hands = result



            if hands is None or len(hands)==0:

                return None,0,None,None



            all_x=[]
            all_y=[]



            for hand in hands:

                for lm in hand["lmList"]:

                    all_x.append(int(lm[0]))
                    all_y.append(int(lm[1]))



            pad = 40


            h,w = img.shape[:2]


            x1 = max(0,min(all_x)-pad)
            y1 = max(0,min(all_y)-pad)

            x2 = min(w,max(all_x)+pad)
            y2 = min(h,max(all_y)+pad)



            crop = img[
                y1:y2,
                x1:x2
            ]



            if crop.size == 0:

                return None,0,None,None



            inp,processed = self.preprocess_image(
                crop
            )


            prediction = self.model.predict(
                inp,
                verbose=0
            )



            index = np.argmax(
                prediction[0]
            )


            confidence = float(
                prediction[0][index]
            )


            label = self.idx_to_class[index]



            # debug top 3

            top3 = np.argsort(
                prediction[0]
            )[-3:][::-1]


            print("\nTop 3 Predictions:")

            for i in top3:

                print(
                    f"{self.idx_to_class[i]} : {prediction[0][i]:.4f}"
                )



            # smoothing

            self.prediction_buffer.append(label)



            if len(self.prediction_buffer) > 5:

                self.prediction_buffer.pop(0)



            stable_label = Counter(
                self.prediction_buffer
            ).most_common(1)[0][0]



            # voice
            self.speak(stable_label)



            return (
                stable_label,
                confidence,
                (x1,y1,x2,y2),
                processed
            )



        except Exception as e:

            print(
                "Prediction error:",
                e
            )

            return None,0,None,None




# ==========================================
# MAIN
# ==========================================

def main():


    cap = cv2.VideoCapture(0)


    cap.set(
        cv2.CAP_PROP_FRAME_WIDTH,
        1280
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        720
    )



    detector = HandSignDetector()



    while True:


        success,img = cap.read()


        if not success:

            continue



        img = cv2.flip(
            img,
            1
        )



        label,confidence,bbox,processed = detector.predict(
            img
        )



        if bbox is not None:


            x1,y1,x2,y2 = bbox


            cv2.rectangle(
                img,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )



        if label is not None:


            # background box

            cv2.rectangle(
                img,
                (img.shape[1]-420,20),
                (img.shape[1]-20,160),
                (0,0,0),
                -1
            )


            cv2.putText(
                img,
                f"Alphabet: {label}",
                (img.shape[1]-390,80),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.3,
                (0,255,0),
                3
            )


            cv2.putText(
                img,
                f"Confidence: {confidence*100:.1f}%",
                (img.shape[1]-390,130),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0,255,255),
                2
            )



        cv2.imshow(
            "Sign Alphabet Recognition",
            img
        )



        if processed is not None:

            cv2.imshow(
                "Model Input",
                processed
            )



        if cv2.waitKey(1) == ord('q'):

            break



    cap.release()

    cv2.destroyAllWindows()



if __name__=="__main__":

    main()

Loading MobileNetV2 model...
Model loaded successfully!

Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}

Top 3 Predictions:
m : 0.9237
o : 0.0643
b : 0.0105

Top 3 Predictions:
w : 0.9579
m : 0.0242
n : 0.0147

Top 3 Predictions:
a : 0.9829
e : 0.0095
x : 0.0073

Top 3 Predictions:
x : 0.6899
a : 0.2542
e : 0.0209

Top 3 Predictions:
e : 0.4322
q : 0.3573
a : 0.1512

Top 3 Predictions:
a : 0.9617
q : 0.0159
x : 0.0137
Voice error: run loop already started

Top 3 Predictions:
a : 0.9947
x : 0.0021
e : 0.0012

Top 3 Predictions:
a : 0.9988
e : 0.0005
m : 0.0002

Top 3 Predictions:
a : 0.9983
b : 0.0007
x : 0.0005

Top 3 Predictions:
a : 0.9385
e : 0.0308
q : 0.0202

Top 3 Predictions:
a : 0.6974
e : 0.2715
q : 0.0282

Top 3 Predictions:
q : 0.4323
a : 0.3397
e : 0.2185

Top 3 Predictions:
a : 0.989